# 03. 기상(ASOS 시간자료) + 태양 위치 + 터널 거리 피처

입력: `data/interim/accident_clean.csv`, `data/interim/tunnel_expressway.csv`, `data/external/asos_stations.csv`
출력: `data/processed/accident_features.csv`

**추가 준비**
1. data.go.kr에서 **15057210 기상청_지상(종관, ASOS) 시간자료 조회서비스** 활용신청 (자동승인, 같은 키 사용)
2. ASOS 관측소 위치 파일을 `data/external/asos_stations.csv`로 저장 (기상자료개방포털 data.kma.go.kr → 지점정보에서 종관(ASOS) 지점 목록 CSV). 지점번호·위도·경도 열만 있으면 됨

만드는 피처
| 구분 | 피처 | 의미 |
|---|---|---|
| 기상 | 일조 `ss`, 일사 `icsr` | 햇빛이 실제로 있었는지, 얼마나 강했는지 |
| 기상 | 전운량 `dc10Tca`, 시정 `vs` | 구름이 해를 가렸는지, 시야 거리 |
| 기상 | 풍속 `ws`, 강수 `rn`, 기온 `ta`, 습도 `hm`, 적설 `dsnw` | 노면·주행 조건 |
| 태양 | 고도 `sun_elev`, 방위각 `sun_azi` | 해가 얼마나 낮게, 어느 쪽에 있는지 (pvlib) |
| 태양 | `glare_low_sun` | 고도 0~25°: 운전자 정면 시야에 들어오는 낮은 해 |
| 터널 | 가장 가까운 터널 입·출구까지 거리, 터널 길이 | 명순응·암순응 구간 여부 |

진행방향 대비 태양 방위각 차이(눈부심의 핵심)는 도로 진행 방위각이 필요해서, 도로 형상 데이터가 정해지면 추가해요.

In [ ]:
%pip install pvlib scikit-learn requests -q

In [ ]:
import os, time
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import pvlib
from sklearn.neighbors import BallTree

ROOT = Path.cwd()
INTERIM, EXTERNAL, PROCESSED = (ROOT / "data" / d for d in ["interim", "external", "processed"])
WEATHER_CACHE = EXTERNAL / "asos_hourly"
for d in [EXTERNAL, PROCESSED, WEATHER_CACHE]:
    d.mkdir(parents=True, exist_ok=True)

acc = pd.read_csv(INTERIM / "accident_clean.csv", parse_dates=["사고일시"])
tun = pd.read_csv(INTERIM / "tunnel_expressway.csv")

def get_service_key():
    key = os.environ.get("DATA_GO_KR_KEY")
    if not key and (ROOT / ".env").exists():
        for line in (ROOT / ".env").read_text(encoding="utf-8").splitlines():
            if line.strip().startswith("DATA_GO_KR_KEY="):
                key = line.split("=", 1)[1].strip().strip('"').strip("'")
    assert key, ".env 에 키가 없습니다"
    return key
SERVICE_KEY = get_service_key()

## 1. 사고 지점마다 가장 가까운 ASOS 관측소 찾기

In [ ]:
st = pd.read_csv(EXTERNAL / "asos_stations.csv", encoding_errors="replace")
# 열 이름이 파일마다 조금씩 달라서 자동으로 찾음
col = lambda key: next(c for c in st.columns if key in c)
st = st.rename(columns={col("지점"): "stn_id", col("위도"): "stn_lat", col("경도"): "stn_lon"})
st = st.drop_duplicates("stn_id", keep="last")[["stn_id", "stn_lat", "stn_lon"]]
print(len(st), "개 관측소")

def nearest(points_lat, points_lon, ref_lat, ref_lon):
    """각 점에서 가장 가까운 ref 점의 위치와 거리(km)"""
    tree = BallTree(np.radians(np.c_[ref_lat, ref_lon]), metric="haversine")
    dist, idx = tree.query(np.radians(np.c_[points_lat, points_lon]), k=1)
    return idx[:, 0], dist[:, 0] * 6371

i, d = nearest(acc["lat"], acc["lon"], st["stn_lat"], st["stn_lon"])
acc["stn_id"] = st["stn_id"].values[i]
acc["stn_dist_km"] = d
print("관측소까지 거리(km):", acc["stn_dist_km"].describe().round(1).to_dict())

## 2. ASOS 시간자료 받기
관측소 × 월 단위로 받아서 `data/external/asos_hourly/`에 저장해요. 한 번 받은 건 다시 안 받아요.
나중에 사고가 없는 시간대(네거티브 샘플)에도 같은 기상자료를 쓰기 때문에 사고 시점만이 아니라 기간 전체를 받아요.
개발계정 하루 10,000회 제한이 있어서, 중간에 멈추면 다음 날 이 셀을 다시 실행하면 이어서 받아요.

In [ ]:
ASOS_URL = "https://apis.data.go.kr/1360000/AsosHourlyInfoService/getWthrDataList"

def fetch_asos_month(stn_id, month_start):
    out = WEATHER_CACHE / f"{stn_id}_{month_start:%Y%m}.csv"
    if out.exists():
        return
    month_end = min(month_start + pd.offsets.MonthEnd(0), pd.Timestamp.today().normalize() - pd.Timedelta(days=1))
    params = {"serviceKey": SERVICE_KEY, "pageNo": 1, "numOfRows": 999, "dataType": "JSON",
              "dataCd": "ASOS", "dateCd": "HR", "stnIds": stn_id,
              "startDt": f"{month_start:%Y%m%d}", "startHh": "00",
              "endDt": f"{month_end:%Y%m%d}", "endHh": "23"}
    for attempt in range(3):
        try:
            r = requests.get(ASOS_URL, params=params, timeout=30)
            body = r.json()["response"]
            break
        except Exception as e:
            if attempt == 2:
                raise RuntimeError(f"{stn_id} {month_start:%Y-%m}: {r.text[:200]}") from e
            time.sleep(2 ** attempt)
    if body["header"]["resultCode"] != "00":
        raise RuntimeError(f"{stn_id} {month_start:%Y-%m}: {body['header']['resultMsg']}")
    pd.DataFrame(body["body"]["items"]["item"]).to_csv(out, index=False, encoding="utf-8-sig")
    time.sleep(0.1)

months = pd.date_range(acc["사고일시"].min().to_period("M").to_timestamp(),
                       acc["사고일시"].max(), freq="MS")
stations = sorted(acc["stn_id"].unique())
print(f"관측소 {len(stations)}개 × {len(months)}개월 = {len(stations) * len(months)}회 호출(캐시 제외)")

for n, stn in enumerate(stations, 1):
    for m in months:
        fetch_asos_month(stn, m)
    print(f"  {n}/{len(stations)} 관측소 {stn} 완료")

In [ ]:
WEATHER_COLS = {"ta": "기온", "rn": "강수", "ws": "풍속", "hm": "습도", "ss": "일조", "icsr": "일사",
                "dc10Tca": "전운량", "vs": "시정", "dsnw": "적설", "ts": "지면온도"}

w = pd.concat([pd.read_csv(f, dtype=str) for f in WEATHER_CACHE.glob("*.csv")], ignore_index=True)
w["tm"] = pd.to_datetime(w["tm"])
w["stn_id"] = w["stnId"].astype(int)
for c in WEATHER_COLS:
    w[c] = pd.to_numeric(w[c], errors="coerce")
w = w[["stn_id", "tm", *WEATHER_COLS]]
w[["rn", "ss", "icsr", "dsnw"]] = w[["rn", "ss", "icsr", "dsnw"]].fillna(0)   # 비어 있으면 '없음'(0)으로 봄
w.to_csv(EXTERNAL / "asos_hourly_all.csv", index=False, encoding="utf-8-sig")
print(w.shape)
w.describe().T.round(2)

## 3. 사고 ↔ 기상 매칭
ASOS의 `tm` 14시 = 13:00~14:00 관측값이에요. 사고가 13:40이면 14시 값을 붙이도록 사고시각을 '다음 정시'로 올려서 맞춰요.

In [ ]:
acc["tm"] = acc["사고일시"].dt.ceil("h")
acc.loc[acc["사고일시"] == acc["tm"], "tm"] += pd.Timedelta(hours=1)   # 정각(분 정보 없음)이면 그 시간대 값
acc = acc.merge(w, on=["stn_id", "tm"], how="left")
print("기상 매칭 실패:", acc["ta"].isna().sum(), "/", len(acc))

## 4. 태양 위치 (pvlib)
사고 시각·위치에서 해의 고도(지평선 위 몇 도)와 방위각(북=0°, 동=90°)을 계산해요.
분 정보가 없으면 그 시간의 30분 시점으로 계산해요.

In [ ]:
t_sun = acc["사고일시"].where(acc["사고일시"].dt.minute > 0, acc["사고일시"] + pd.Timedelta(minutes=30))
times = pd.DatetimeIndex(t_sun).tz_localize("Asia/Seoul")
sp = pvlib.solarposition.get_solarposition(times, acc["lat"].values, acc["lon"].values)
acc["sun_elev"] = sp["apparent_elevation"].values
acc["sun_azi"] = sp["azimuth"].values
acc["is_daylight"] = acc["sun_elev"] > 0
acc["glare_low_sun"] = acc["sun_elev"].between(0, 25)
print(acc[["sun_elev", "sun_azi"]].describe().round(1))
print("낮은 해(0~25°) 시간대 사고 비율:", acc["glare_low_sun"].mean().round(3))

## 5. 터널 입·출구까지 거리
터널 입구·출구 좌표를 모두 '터널 끝점'으로 놓고 가장 가까운 끝점까지 거리를 구해요.
(어느 쪽이 진입이고 어느 쪽이 진출인지는 주행방향이 정리되면 나눠요.)

In [ ]:
ends = pd.concat([
    tun[["tunnel_id", "start_lat", "start_lon", "length_m"]].set_axis(["tunnel_id", "lat", "lon", "length_m"], axis=1),
    tun[["tunnel_id", "end_lat", "end_lon", "length_m"]].set_axis(["tunnel_id", "lat", "lon", "length_m"], axis=1),
], ignore_index=True)
i, d = nearest(acc["lat"], acc["lon"], ends["lat"], ends["lon"])
acc["tunnel_id"] = ends["tunnel_id"].values[i]
acc["tunnel_end_dist_km"] = d
acc["tunnel_length_m"] = ends["length_m"].values[i]
acc["near_tunnel_1km"] = acc["tunnel_end_dist_km"] <= 1
print("터널 끝점 1km 이내 사고 비율:", acc["near_tunnel_1km"].mean().round(3))

In [ ]:
acc.to_csv(PROCESSED / "accident_features.csv", index=False, encoding="utf-8-sig")
print(acc.shape)
acc.head()